In [3]:


import nltk
import pandas as pd
from collections import Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

print("Libraries imported successfully.")


Libraries imported successfully.


In [4]:
# Download tokenizer resources used for the contextual examples.

nltk.download('punkt')
nltk.download('punkt_tab')

print("NLTK tokenizer resources are ready.")


NLTK tokenizer resources are ready.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mahat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mahat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
# Q4(a)-(b): Ambiguity examples and intended meanings.
#
# Each example uses context to determine the intended sense of an ambiguous word.

ambiguity_examples = [
    {
        "Sentence": "I deposited my salary at the bank.",
        "Ambiguous Word": "bank",
        "Intended Meaning": "A financial institution"
    },
    {
        "Sentence": "The fisherman sat on the bank of the river.",
        "Ambiguous Word": "bank",
        "Intended Meaning": "The land beside a river"
    },
    {
        "Sentence": "The baseball player swung the bat.",
        "Ambiguous Word": "bat",
        "Intended Meaning": "A piece of sports equipment"
    },
    {
        "Sentence": "A bat flew out of the cave at night.",
        "Ambiguous Word": "bat",
        "Intended Meaning": "A flying mammal"
    },
    {
        "Sentence": "Please turn on the light in the room.",
        "Ambiguous Word": "light",
        "Intended Meaning": "Illumination"
    },
    {
        "Sentence": "This bag is very light to carry.",
        "Ambiguous Word": "light",
        "Intended Meaning": "Not heavy"
    },
    {
        "Sentence": "She lit a match to start the fire.",
        "Ambiguous Word": "match",
        "Intended Meaning": "A small stick used to produce fire"
    },
    {
        "Sentence": "The two colors are a perfect match.",
        "Ambiguous Word": "match",
        "Intended Meaning": "A correspondence or similarity"
    }
]

ambiguity_df = pd.DataFrame(ambiguity_examples)

print("Ambiguity examples:")
display(ambiguity_df)


Ambiguity examples:


,Sentence,Ambiguous Word,Intended Meaning
0,I deposited my salary at the bank.,bank,A financial institution
1,The fisherman sat on the bank of the river.,bank,The land beside a river
2,The baseball player swung the bat.,bat,A piece of sports equipment
3,A bat flew out of the cave at night.,bat,A flying mammal
4,Please turn on the light in the room.,light,Illumination
5,This bag is very light to carry.,light,Not heavy
6,She lit a match to start the fire.,match,A small stick used to produce fire
7,The two colors are a perfect match.,match,A correspondence or similarity


In [6]:
# Q4(c): Coreference examples and their antecedents.

coreference_examples = [
    {
        "Sentence": "Ravi went to the library because he needed a book.",
        "Referring Expression": "he",
        "Antecedent": "Ravi"
    },
    {
        "Sentence": "The students submitted their assignments before the deadline.",
        "Referring Expression": "their",
        "Antecedent": "The students"
    },
    {
        "Sentence": "Maria bought a laptop because it was on sale.",
        "Referring Expression": "it",
        "Antecedent": "a laptop"
    },
    {
        "Sentence": "The dog chased the ball until it reached the garden.",
        "Referring Expression": "it",
        "Antecedent": "the ball"
    },
    {
        "Sentence": "John called David after he finished the project.",
        "Referring Expression": "he",
        "Antecedent": "John (intended reading)"
    }
]

coreference_df = pd.DataFrame(coreference_examples)

print("Coreference examples:")
display(coreference_df)


Coreference examples:


,Sentence,Referring Expression,Antecedent
0,Ravi went to the library because he needed a b...,he,Ravi
1,The students submitted their assignments befor...,their,The students
2,Maria bought a laptop because it was on sale.,it,a laptop
3,The dog chased the ball until it reached the g...,it,the ball
4,John called David after he finished the project.,he,John (intended reading)


In [10]:
from pathlib import Path
from sklearn.datasets import load_files

train_dir = Path("../data/20news-bydate-train")

newsgroups = load_files(
    train_dir,
    categories=["sci.space", "comp.graphics"],
    encoding="latin1",
    decode_error="ignore",
    shuffle=True,
    random_state=42
)

space_data = [
    text for text, label in zip(newsgroups.data, newsgroups.target)
    if newsgroups.target_names[label] == "sci.space"
]

graphics_data = [
    text for text, label in zip(newsgroups.data, newsgroups.target)
    if newsgroups.target_names[label] == "comp.graphics"
]

print("sci.space documents:", len(space_data))
print("comp.graphics documents:", len(graphics_data))


sci.space documents: 593
comp.graphics documents: 584


In [12]:
stop_words = set(CountVectorizer(stop_words='english').get_stop_words())

def frequent_terms(documents, top_n=20):
    word_counts = Counter()

    for document in documents:
        words = nltk.word_tokenize(document.lower())

        for word in words:
            if (
                word.isalpha()
                and len(word) > 2
                and word not in stop_words
            ):
                word_counts[word] += 1

    return word_counts.most_common(top_n)


space_terms = frequent_terms(space_data, 20)
graphics_terms = frequent_terms(graphics_data, 20)

space_df = pd.DataFrame(space_terms, columns=["Term", "Frequency"])
graphics_df = pd.DataFrame(graphics_terms, columns=["Term", "Frequency"])

print("Top 20 domain-specific terms – sci.space:")
display(space_df)

print("Top 20 domain-specific terms – comp.graphics:")
display(graphics_df)

Top 20 domain-specific terms – sci.space:


,Term,Frequency
0,space,1193
1,subject,633
2,organization,614
3,lines,613
4,writes,452
5,nasa,409
6,article,385
7,just,342
8,launch,328
9,like,324


Top 20 domain-specific terms – comp.graphics:


,Term,Frequency
0,lines,641
1,subject,625
2,organization,571
3,image,517
4,graphics,432
5,university,356
6,file,289
7,files,265
8,does,259
9,use,258


In [13]:
# Combine the two domain vocabularies and display at least 20 terms.

combined_terms = Counter(dict(space_terms)) + Counter(dict(graphics_terms))

combined_df = pd.DataFrame(
    combined_terms.most_common(20),
    columns=["Domain Term", "Combined Frequency"]
)

print("At least 20 frequent domain-specific terms:")
display(combined_df)


At least 20 frequent domain-specific terms:


,Domain Term,Combined Frequency
0,subject,1258
1,lines,1254
2,space,1193
3,organization,1185
4,writes,697
5,university,625
6,article,602
7,like,545
8,image,517
9,data,474
